In [5]:
!pip install langchain-huggingface

In [6]:
import os
from google.colab import userdata

# 1. Get the key from Colab Secrets
api_key = userdata.get("APIKEY")

# 2. Set it as an environment variable (Crucial step)
os.environ["HUGGINGFACEHUB_API_TOKEN"] = api_key



In [7]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from dotenv import load_dotenv
from typing import TypedDict

In [8]:
load_dotenv()

model =HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    huggingfacehub_api_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
)

llm = ChatHuggingFace(llm = model)


In [9]:
llm.invoke("hey")

AIMessage(content="How's it going? Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 36, 'total_tokens': 57}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': 'fp_f613d2b18eccee549c5f', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e24fa-d2f6-7ee3-9695-88a7f91cb283-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 21, 'total_tokens': 57})

In [22]:
class LLMstate(TypedDict):
    question:str
    answer:str

In [28]:
def llm_qa(state:LLMstate):
    question = state["question"]
    prompt = f"Answer the following {question}"
    answer = llm.invoke(prompt).content
    state["answer"] = answer

    return state


In [29]:
graph = StateGraph(LLMstate)

In [30]:
graph.add_node("llm_qa",llm_qa)

graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

In [31]:
workflow = graph.compile()

In [32]:
initial_state = {'question': 'How far is moon from the earth?'}
final_state = workflow.invoke(initial_state)

In [33]:
final_state


{'question': 'How far is moon from the earth?',
 'answer': "The average distance from the Earth to the Moon is approximately 384,400 kilometers (238,900 miles). However, this distance varies slightly due to the elliptical shape of the Moon's orbit around the Earth.\n\nAt its closest point (called perigee), the Moon is about 363,300 kilometers (225,300 miles) away from the Earth.\n\nAt its farthest point (called apogee), the Moon is about 405,500 kilometers (252,000 miles) away from the Earth.\n\nSo, the distance from the Earth to the Moon is not a fixed value but rather an average distance with slight variations due to the Moon's elliptical orbit."}